In [2]:
import os
os.environ["OPENAI_API_KEY"] = ""


In [2]:
from phi.tools import Toolkit
from typing import Dict, Any, Optional
import requests
import json

class MCPRestTool(Toolkit):
    """
    MCP REST client that allows PhiData Agent to call tools from MCP server.
    Example endpoints: /mongo_create, /mongo_read, /mongo_update, /mongo_delete
    """

    def __init__(self, base_url: str, api_key: Optional[str] = None):
        super().__init__(name="mcp_rest_tool")
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key

        # Register callable functions
        self.register(self.list_tools)
        self.register(self.call_tool)

    # ---------------------------
    # Helper methods
    # ---------------------------
    def _headers(self) -> Dict[str, str]:
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            headers["Authorization"] = f"Bearer {self.api_key}"
        return headers

    # ---------------------------
    # Registered tool methods
    # ---------------------------
    def list_tools(self) -> str:
        """List all available MCP endpoints."""
        try:
            resp = requests.get(f"{self.base_url}/tools", headers=self._headers(), timeout=10)
            if resp.status_code == 200:
                tools = resp.json().get("tools", [])
                return "Available MCP tools: " + ", ".join(tools)
            return f"Could not fetch tools (HTTP {resp.status_code})."
        except Exception as e:
            return f"Error while listing tools: {e}"

    def call_tool(self, action: str, payload: Optional[Dict[str, Any]] = None) -> str:
        """
        Call an MCP tool endpoint with given action and payload.
        Example: call_tool(action='mongo_read', payload={'filter': {'city': 'DELHI'}})
        """
        payload = payload or {}

        try:
            url = f"{self.base_url}/tools/{action}"
            response = requests.post(url, headers=self._headers(), json=payload, timeout=15)
            if response.status_code == 200:
                return json.dumps(response.json(), indent=2)
            else:
                return f"HTTP {response.status_code}: {response.text}"
        except Exception as e:
            return f"Error calling {action}: {e}"


In [20]:
from phi.agent import Agent
from phi.llm.openai import OpenAIChat
#from tools.mcp_rest_tool import MCPRestTool

# Create MCP REST Tool
mcp_tool = MCPRestTool(base_url="http://localhost:5000", api_key="ABCD123456")  # or provide your key

# Create Agent
agent = Agent(
    name="MCP Client Agent",
    description="Agent that discovers and calls tools from MCP server at http://localhost:5000",
    llm=OpenAIChat(model="gpt-4o-mini"),
    tools=[mcp_tool],
    show_tool_calls=True,
    markdown=True,
)

# Run test interactions
#agent.print_response("List all available MCP tools.")
agent.print_response("Use mongo_read tool to Get all documents from 'BestBuy' purchase_showroom and city 'Hyderabad' and cost more than 10000 and shipping cost more than 1000",stream=True)


Output()